# Skeleton-only Spatial-GCN SHuBERT pretraining
Run all cells with a GPU runtime. This trains one shard per epoch, downloads a resumable checkpoint bundle after every shard, and deletes only the completed local shard. Input is skeleton coordinates only; no RGB, gloss, translation text, or lexical boundaries are used.

In [ ]:
import os, shutil, subprocess
PROJECT = '/content/youtube-asl-skeleton-bert'
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'agent/shape-aware-stgcn', 'https://github.com/ss-sebastian/youtube-asl-skeleton-bert.git', PROJECT], check=True)


In [ ]:
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2'], check=True)
subprocess.run(['pip', 'install', '-q', 'scikit-learn>=1.4'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', PROJECT], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then Run all again.'
print(torch.cuda.get_device_name(0))


In [ ]:
import queue, threading, time
command = ['python', '-u', f'{PROJECT}/scripts/colab_spatial_shubert_full_train.py']
print('Starting:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command, cwd=f'{PROJECT}/scripts', stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1,
)
messages = queue.Queue()
def forward_output():
    for line in process.stdout:
        messages.put(line)
    messages.put(None)
threading.Thread(target=forward_output, daemon=True).start()
started = time.monotonic()
finished_output = False
while process.poll() is None or not finished_output:
    try:
        message = messages.get(timeout=20)
        if message is None:
            finished_output = True
        else:
            marker = 'COLAB_DOWNLOAD='
            if message.strip().startswith(marker):
                from google.colab import files
                bundle = message.strip()[len(marker):]
                print(f'Initiating browser download: {bundle}', flush=True)
                files.download(bundle)
                print('Checkpoint download initiated; continuing to the next shard.', flush=True)
            else:
                print(message, end='', flush=True)
    except queue.Empty:
        elapsed = (time.monotonic() - started) / 60
        print(f'[launcher heartbeat] still running; elapsed={elapsed:.1f} min', flush=True)
returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print('Spatial-SHuBERT training launcher completed successfully.', flush=True)
